# Usage | 3. Existing bike network
This notebook explains how growbikenet can extend an existing bike network.

**Parameters covered**: `existing_network_spacing`

We start every Usage notebook with the standard way of importing growbikenet:

In [ ]:
import growbikenet as gbn

So far growbikenet was executed with the default parameter setting `existing_network_spacing=None`, which instructed growbikenet to ignore existing bicycle infrastructure. This works for most cities, as existing infrastructure is usually negligible and one might as well just start from scratch. However, there are some cities with an already existing substantial network which would be useful to incorporate into the growth process. By calling growbikenet with the parameter `existing_network_spacing='auto'` or with a positive integer, it will do exactly that.

In this case, the process of generating seed points is amended:
- Consider all network components of the existing bike network that have a minimum length. This ensures that tiny, insignificant pieces are ignored.
- On these components, choose a random first seed point.
- Choose the closest seed point on the components that is at least `existing_network_spacing` meters away. The `'auto'` option automatically chooses a recommended distance, at 50% of the `seed_point_grid_spacing`.
- Proceed with the previous step until no more seed points can be placed on the components.
- Now generate all the other seed points as usual, but do not consider seed points that are too close to already existing seed points.

Let us run growbikenet on Athens with the `existing_network_spacing='auto'` option and observe the results:

In [ ]:
edges_ranked = gbn.growbikenet("Municipality of Athens",
                               existing_network_spacing='auto',)

The existing bike network is saved as multilinestring into the first row of the resulting geodataframe with several entries being `None`:

In [ ]:
edges_ranked.head()

To visualize the outcome, we plot first the existing bike network (first row) in blue, then the grown network (all other rows) in green. To add layer control in the top right of the map, we import folium:

In [ ]:
import folium
viz = edges_ranked.iloc[:1].explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 2, "color": "#9999cc"},
                        name="Existing bike network")
viz = edges_ranked.iloc[1:].explore(m=viz, 
                     style_kwds={"weight": 3, "color": "#096a51"},
                        name="Grown bike network")
folium.LayerControl().add_to(viz)
viz

Note how the short existing pieces in the northeast are ignored, but the other big enough components are incorporated into the growth process.

Let us add the outcome from growth from scratch (without the existing network) in orange to see the difference:

In [ ]:
edges_ranked_from_scratch = gbn.growbikenet("Municipality of Athens")

In [ ]:
viz = edges_ranked.iloc[:1].explore(tiles="CartoDB Positron",
                     style_kwds={"weight": 2, "color": "#9999cc"},
                        name="Existing bike network")
viz = edges_ranked_from_scratch.explore(m=viz, 
                     style_kwds={"weight": 3, "color": "#f19730"},
                        name="Grown bike network (from scratch)")
viz = edges_ranked.iloc[1:].explore(m=viz, 
                     style_kwds={"weight": 3, "color": "#096a51"},
                        name="Grown bike network (with existing network)")
folium.LayerControl().add_to(viz)
viz

In general, the network which accounts for existing infrastructure will be longer than the one grown from scratch, in this case

In [ ]:
int((edges_ranked.iloc[-1].length_cumulative-
    edges_ranked.iloc[0].length_cumulative)/1000)

kilometers compared to

In [ ]:
int((edges_ranked_from_scratch.iloc[-1].length_cumulative)/1000)

kilometers, as seed points are generated more densely.